In [3]:
import os

In [4]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\huggingface-text-summarization\\notebooks'

In [5]:
os.chdir("../")
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\huggingface-text-summarization'

In [14]:
import os
from dataclasses import dataclass
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoTokenizer
from src.huggingface_text_summarization.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.huggingface_text_summarization.utils.common import read_yaml,create_directories
from src.huggingface_text_summarization.logging import logger

In [10]:
@dataclass
class DataTransformationConfig:
    root_directory: Path
    data_path: Path
    tokenizer_name: str

In [11]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH, param_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(path_to_yaml=config_file_path)
        self.params = read_yaml(path_to_yaml=param_file_path)

        create_directories(path_to_directories=[self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories(path_to_directories=[config.root_directory])

        data_transformation_config = DataTransformationConfig(
            root_directory=config.root_directory,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name,
        )

        return data_transformation_config

In [15]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        # Tokenize input dialogue
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=1024,
            truncation=True,
            padding="max_length"
        )

        # Tokenize target summary
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"],
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features, batched=True
        )
        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_directory, "samsum_dataset")
        )
    

In [16]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)

data_transformation.convert()

[2026-03-18 15:46:26,130: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-18 15:46:26,137: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-18 15:46:26,138: INFO: common: created directory at: artifacts]
[2026-03-18 15:46:26,139: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-18 15:46:26,809: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-18 15:46:27,175: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-18 15:46:27,769: INFO: _client: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]


c:\Users\SAAD TARIQ\github_repositories\students-performance-indicator\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SAAD TARIQ\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[2026-03-18 15:46:28,191: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-18 15:46:28,541: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-03-18 15:46:28,896: INFO: _client: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-03-18 15:46:29,256: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-03-18 15:46:29,604: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1.

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 136326.96 examples/s]
